# 3. Model train + SHAP/FFA analysis

**Purpose:** Run pipeline (model data → PGx → final model), then SHAP and FFA analysis and combine results for the Causal tab. No build or deploy here.

**Flow:** Run this notebook first. Then [4_dashboard_visuals.ipynb](4_dashboard_visuals.ipynb) (BupaR, DTW, FP-Growth). Then [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb).

**Steps:** Sync inputs → Verify → Pipeline Phase 4 (model data) → Phase 5 (PGx) → Phase 6 (final model) → Step 1a (metadata) → Step 7 (SHAP) → Step 8 (FFA) → Combine (SHAP+FFA to dashboard) → optional inspection.

**Memory:** Pipeline scripts use **DuckDB and Parquet** where possible for efficient memory use (Step 4 model data, Step 6/7 SHAP/FFA data prep, combine); pandas is used only where required (e.g. model/SHAP APIs). See project `.cursorrules` for data-processing preferences.\n\nPrerequisites: Cohorts (Step 2), feature importance (Step 3/3b). Run from repo root.

In [4]:
# Setup: paths and project root
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "10_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "10_risk_dashboard").exists():
    pass
else:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import sync_s3_to_local, check_step_checkpoint_exists, save_step_checkpoint

DASHBOARD_DIR = PROJECT_ROOT / "10_risk_dashboard"
DATA_PREP_DIR = DASHBOARD_DIR / "data_preparation"
DEPLOY_DIR = DASHBOARD_DIR / "deployment"
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

print("PGx Risk Calculator Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dashboard dir: {DASHBOARD_DIR}")
print(f"Data prep: {DATA_PREP_DIR}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print("=" * 60)

PGx Risk Calculator Workflow
Project root: /home/pgx3874/pgx-analysis
Dashboard dir: /home/pgx3874/pgx-analysis/10_risk_dashboard
Data prep: /home/pgx3874/pgx-analysis/10_risk_dashboard/data_preparation
Data root (NVMe/local): /mnt/nvme


In [ ]:
# Both cohorts use full age band set
from py_helpers.constants import REQUIRED_COHORTS

# Input dirs (required for pipeline Step 4–6)
# Cohorts: Step 2 cohort.parquet files (create_model_data reads case/control and target dates from here).
COHORTS_ROOT = DATA_ROOT / "gold" / "cohorts"
# Feature importance: Step 3/3b outputs — cohort_feature_importance.csv and feature_filtering_summary.json per cohort/age_band.
FI_ROOT = DATA_ROOT / "gold" / "feature_importance"
STEP3_OUTPUTS = STEP3B_OUTPUTS = FI_ROOT
# Model data: single canonical location (Step 4 output, Step 5/6 input).
from py_helpers.env_utils import get_model_data_root
MODEL_DATA_ROOT = get_model_data_root()

# Output dirs (Step 6 final model outputs; data prep and Lambda read from these)
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_OUTPUTS_ALT = DATA_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_GOLD = DATA_ROOT / "gold" / "final_model"  # S3 layout: cohort/13-24/*.joblib

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("\nInput dirs (for Step 4–6):")
print(f"  Cohorts (2):              {COHORTS_ROOT}")
print(f"  Feature importance (3/3b): {FI_ROOT}  (CSVs + feature_filtering_summary.json)")
print(f"  Model data (4; in/out):   {MODEL_DATA_ROOT}")
print("\nOutput dirs (Step 6):")
print(f"  Project:   {FINAL_MODEL_OUTPUTS}")
print(f"  NVMe:      {FINAL_MODEL_OUTPUTS_ALT}")
print(f"  gold/NVMe: {FINAL_MODEL_GOLD}")

Cohorts and age bands:
  opioid_ed: ['13-24', '25-44', '45-54', '55-64']
  non_opioid_ed: ['65-74', '75-84', '85-94']

Input dirs (for Step 4–6):
  Cohorts (2):              /mnt/nvme/gold/cohorts
  Feature importance (3/3b): /mnt/nvme/gold/feature_importance  (CSVs + feature_filtering_summary.json)
  Model data (4; in/out):   /mnt/nvme/4_model_data

Output dirs (Step 6):
  Project:   /home/pgx3874/pgx-analysis/6_final_model/outputs
  NVMe:      /mnt/nvme/6_final_model/outputs
  gold/NVMe: /mnt/nvme/gold/final_model


## Clear all checkpoints and pipeline outputs (optional — for a fresh run)

Run this cell **once** when you want to rebuild the full pipeline from Step 4 through SHAP/FFA from scratch. It (1) clears S3 **checkpoints** (pgx-repository: 4_model_data, 6_final_model, 9_dashboard_metadata), (2) deletes S3 **pipeline outputs** (pgxdatalake: `gold/cohorts_model_data/`, `gold/final_model/`) so Step 4 and Step 6 re-run instead of re-downloading, (3) removes local output directories. After this, run the Sync cell and then Steps 4 → 5 → 6 → 1a → Step 7 → Step 8 → Combine.

In [6]:
# Clear S3 checkpoints, S3 pipeline outputs, and local outputs for a fresh model + SHAP/FFA run.
import shutil
import subprocess
from py_helpers.workflow_sync_checkpoint import clear_step_checkpoints, delete_step_checkpoint

# 1) S3 checkpoint metadata (pgx-repository) so steps don't think they're done
for step in ("4_model_data", "6_final_model"):
    for cohort, bands in REQUIRED_COHORTS.items():
        n = clear_step_checkpoints(step, cohort, bands, logger=None)
        print(f"Cleared {n} checkpoint(s) for {step} / {cohort}")
delete_step_checkpoint("9_dashboard_metadata", "all", "all", logger=None)
print("Cleared checkpoint 9_dashboard_metadata (all/all)")

# 2) S3 pipeline outputs (pgxdatalake) so Step 4 and Step 6 re-run instead of re-downloading
_aws = shutil.which("aws") or "aws"
_profile = ["--profile", AWS_PROFILE] if AWS_PROFILE else []
for prefix in ("gold/cohorts_model_data/", "gold/final_model/"):
    uri = f"s3://{S3_BUCKET}/{prefix}"
    r = subprocess.run([_aws, "s3", "rm", uri, "--recursive"] + _profile, capture_output=True, text=True)
    if r.returncode == 0:
        print(f"Cleared S3 {uri}")
    else:
        print(f"S3 rm {uri}: exit {r.returncode} (check credentials); {r.stderr or r.stdout or ''}")

# 3) Local output directories
dirs_to_clear = [
    MODEL_DATA_ROOT,
    FINAL_MODEL_OUTPUTS,
    FINAL_MODEL_OUTPUTS_ALT,
    PROJECT_ROOT / "7_shap_analysis" / "outputs",
    PROJECT_ROOT / "8_ffa_analysis" / "outputs",
    PROJECT_ROOT / "10_risk_dashboard" / "outputs",
]
for d in dirs_to_clear:
    d = Path(d)
    if d.exists():
        shutil.rmtree(d)
        print(f"Removed {d}")
    else:
        print(f"(skip, not present) {d}")
print("Done. Re-run Sync and then Steps 4 → 5 → 6 → 1a → Step 7 → Step 8 → Combine for a fresh model and SHAP/FFA outputs.")

Cleared 4 checkpoint(s) for 4_model_data / opioid_ed
Cleared 3 checkpoint(s) for 4_model_data / non_opioid_ed
Cleared 4 checkpoint(s) for 6_final_model / opioid_ed
Cleared 3 checkpoint(s) for 6_final_model / non_opioid_ed
Cleared checkpoint 9_dashboard_metadata (all/all)
Cleared S3 s3://pgxdatalake/gold/cohorts_model_data/
Cleared S3 s3://pgxdatalake/gold/final_model/
Removed /mnt/nvme/4_model_data
Removed /home/pgx3874/pgx-analysis/6_final_model/outputs
(skip, not present) /mnt/nvme/6_final_model/outputs
Removed /home/pgx3874/pgx-analysis/7_shap_analysis/outputs
Removed /home/pgx3874/pgx-analysis/8_ffa_analysis/outputs
Removed /home/pgx3874/pgx-analysis/10_risk_dashboard/outputs
Done. Re-run Sync and then Steps 4 → 5 → 6 → 1a → Step 7 → Step 8 → Combine for a fresh model and SHAP/FFA outputs.


## Sync required inputs from S3 to NVMe (idempotent)

Sync **cohorts** (Step 2), **feature importance** (Step 3/3b), and **Step 6** final model outputs from S3 so pipeline and data preparation can read from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [7]:
# Sync cohorts (Step 2), Step 3a/3b feature importance, and Step 6 final models from S3 to NVMe (DATA_ROOT).
# Cohorts -> COHORTS_ROOT (gold/cohorts); Feature importance -> gold/feature_importance; Step 6 -> gold/final_model.
COHORTS_ROOT.mkdir(parents=True, exist_ok=True)
FI_SYNC_TARGET = DATA_ROOT / "gold" / "feature_importance"
FI_SYNC_TARGET.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_GOLD.mkdir(parents=True, exist_ok=True)

sync_s3_to_local(f"s3://{S3_BUCKET}/gold/cohorts/", COHORTS_ROOT, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/feature_importance/", FI_SYNC_TARGET, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/final_model/", FINAL_MODEL_GOLD, profile=AWS_PROFILE)
print("Sync complete. Run Step 0 verification below.")

Sync complete. Run Step 0 verification below.


## Step 0: Verify inputs (FI required; 4_model_data and Step 6 informational)

**Required:** **Feature importance** (Step 3/3b) — must exist for each cohort/age_band so Pipeline Step 4 can run.

**Informational:** **ModelData** checks `DATA_ROOT/4_model_data` and `PROJECT_ROOT/4_model_data` (same location `create_model_data.py` writes to). **Model** = Step 6 outputs. Both are produced by Pipeline Step 4–6 cells below; if already present, you can skip those cells.

In [ ]:
def check_feature_importance(cohort: str, age_band: str) -> bool:
    """Check if feature importance exists using FileResolver pattern."""
    from py_helpers.file_resolver import FileResolver
    # Check Step 3b refined cohort feature importance first
    resolver_3b = FileResolver(
        file_type="cohort_feature_importance",
        project_root=PROJECT_ROOT,
        cohort=cohort,
        age_band=age_band,
        auto_download=False
    )
    if resolver_3b.exists():
        return True
    # Fallback to Step 3a aggregated feature importance
    resolver_3a = FileResolver(
        file_type="aggregated_feature_importance",
        project_root=PROJECT_ROOT,
        cohort=cohort,
        age_band=age_band,
        auto_download=False
    )
    return resolver_3a.exists()

def check_cohorts(cohort: str, age_band: str) -> bool:
    """Check Step 2 cohort.parquet exists for at least one year (2016–2019). Layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet."""
    for year in (2016, 2017, 2018, 2019):
        p = COHORTS_ROOT / f"cohort_name={cohort}" / f"event_year={year}" / f"age_band={age_band}" / "cohort.parquet"
        if p.exists():
            return True
    return False

def check_model_data(cohort: str, age_band: str) -> bool:
    """Check model_events.parquet at canonical MODEL_DATA_ROOT (same location create_model_data.py writes to)."""
    p = MODEL_DATA_ROOT / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
    return p.exists()

def check_final_model(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # 1) Project or DATA_ROOT/6_final_model/outputs: cohort/13_24/models/*.joblib
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        model_dir = base / cohort / ab
        if not model_dir.exists():
            continue
        models_sub = model_dir / "models"
        if models_sub.exists() and any(models_sub.glob("*.joblib")):
            return True
        if (model_dir / "feature_schema.json").exists():
            return True
    # 2) DATA_ROOT/gold/final_model (S3-synced): cohort/13-24/*.joblib (hyphen in age_band)
    gold_dir = FINAL_MODEL_GOLD / cohort / age_band
    if gold_dir.exists() and any(gold_dir.glob("*.joblib")):
        return True
    return False

print("Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)")
print("  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs")
fi_ok_all = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        cohorts_ok = check_cohorts(cohort, age_band)
        fi_ok = check_feature_importance(cohort, age_band)
        model_data_ok = check_model_data(cohort, age_band)
        model_ok = check_final_model(cohort, age_band)
        if not fi_ok:
            fi_ok_all = False
        status = "ready" if fi_ok else "missing FI"
        print(f"  {cohort} / {age_band}:  Cohorts={cohorts_ok}, FI={fi_ok}, ModelData={model_data_ok}, Model={model_ok}  -> {status}")
if fi_ok_all:
    print("\nAll prerequisites are available to build model data. Run Pipeline Step 4–6 cells below.")
    print("  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)")
else:
    print("\nMissing feature importance for some cohort/age_band. Sync from S3 or run Step 3/3b first, then re-run this cell.")
if fi_ok_all:
    cohorts_missing = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands if not check_cohorts(c, ab)]
    if cohorts_missing:
        print("\nCohorts=False for some cohort/age_band. Sync gold/cohorts from S3 (run Sync cell) or run Step 2. Expected layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016–2019).")


Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)
  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs
  opioid_ed / 13-24:  Cohorts=True, FI=True, ModelData=False, Model=True  -> ready
  opioid_ed / 25-44:  Cohorts=True, FI=True, ModelData=False, Model=True  -> ready
  opioid_ed / 45-54:  Cohorts=True, FI=True, ModelData=False, Model=True  -> ready
  opioid_ed / 55-64:  Cohorts=True, FI=True, ModelData=False, Model=True  -> ready
  non_opioid_ed / 65-74:  Cohorts=True, FI=True, ModelData=False, Model=True  -> ready
  non_opioid_ed / 75-84:  Cohorts=True, FI=True, ModelData=False, Model=True  -> ready
  non_opioid_ed / 85-94:  Cohorts=True, FI=True, ModelData=False, Model=True  -> ready

All prerequisites are available to build model data. Run Pipeline Step 4–6 cells below.
  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)


# Pipeline Phase 4: Model data

Build `model_events.parquet` for each cohort/age_band from Step 2 cohort data and Step 3b feature importance. Outputs go to `MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet`. Run the cell below for all PGx cohorts/age_bands defined in this notebook.

In [9]:
# Pipeline Step 4: BUILD model_events.parquet by running create_model_data.py, then QA.
# The script READS: COHORTS_ROOT (cohort.parquet), gold/medical, gold/pharmacy, and feature importance.
# It WRITES: MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet
import duckdb

def _model_data_candidates(cohort: str, age_band: str):
    """Canonical location for model_events.parquet (Step 4 writes to MODEL_DATA_ROOT)."""
    return [MODEL_DATA_ROOT]

def _model_data_path(cohort: str, age_band: str) -> Path:
    """Resolve model_events.parquet path (Step 4 writes to get_model_data_root() = DATA_ROOT or PROJECT on Linux)."""
    for base in _model_data_candidates(cohort, age_band):
        p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
        if p.exists():
            return p
    return None

def _log_model_data_qa(cohort: str, age_band: str) -> None:
    """Log location, target distribution, and control:case ratio for model_events.parquet."""
    path = _model_data_path(cohort, age_band)
    if not path:
        print(f"  [WARN] model_events.parquet not found for {cohort}/{age_band}")
        for base in _model_data_candidates(cohort, age_band):
            p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
            print(f"    Checked: {p}  (exists: {p.exists()})")
        print(f"    Build did not write output. Check script stdout above: [INFO] data roots and example cohort path (exists=?). Layout must be {COHORTS_ROOT}/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016–2019). Sync cohorts to COHORTS_ROOT if needed, then re-run this cell.")
        return
    print(f"  Location: {path}")
    con = duckdb.connect()
    try:
        dist = con.execute("SELECT target, COUNT(*)::BIGINT AS n FROM read_parquet(?) GROUP BY target ORDER BY target", [str(path)]).fetchall()
        total = sum(row[1] for row in dist)
        by_target = {int(row[0]): int(row[1]) for row in dist}
        n_controls = by_target.get(0, 0)
        n_cases = by_target.get(1, 0)
        ratio = (n_controls / n_cases) if n_cases else 0
        print(f"  Target distribution: {by_target} (total rows: {total:,})")
        print(f"  Control:case ratio: {n_controls:,}:{n_cases:,} = {ratio:.2f}:1")
    finally:
        con.close()

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 4: {cohort} / {age_band} (building model_events.parquet)")
        r = subprocess.run(
            [sys.executable, "create_model_data.py", "--cohort", cohort, "--age-band", age_band],
            cwd=PROJECT_ROOT / "4_model_data",
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
        _log_model_data_qa(cohort, age_band)
print("Step 4 complete.")

→ Step 4: opioid_ed / 13-24 (building model_events.parquet)
[INFO] Found Step 3b refined feature importance: /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outputs/opioid_ed/13_24/opioid_ed_13_24_cohort_feature_importance.csv
[INFO] Cohort root: tried ['/mnt/nvme/gold/cohorts', '/mnt/nvme/data/gold_cohorts', '/home/pgx3874/pgx-analysis/data/gold_cohorts'] -> using /mnt/nvme/gold/cohorts (first existing)
[INFO] Medical root: tried ['/mnt/nvme/gold/medical', '/mnt/nvme/data/gold_medical', '/home/pgx3874/pgx-analysis/data/gold_medical'] -> using /mnt/nvme/gold/medical (first existing)
[INFO] Pharmacy root: tried ['/mnt/nvme/gold/pharmacy', '/mnt/nvme/data/gold_pharmacy', '/home/pgx3874/pgx-analysis/data/gold_pharmacy'] -> using /mnt/nvme/gold/pharmacy (first existing)
[INFO] Step 4 data roots: cohorts=/mnt/nvme/gold/cohorts, medical=/mnt/nvme/gold/medical, pharmacy=/mnt/nvme/gold/pharmacy
[INFO] Example cohort path (must exist for build to run): /mnt/nvme/gold/cohorts/cohort_name=op

# Pipeline Phase 5: PGx analysis

Add PGx features (e.g. CPIC drug counts) to model data. Reads from Step 4 outputs and writes updated model data used by Step 6. Run for each cohort/age_band.

In [10]:
# Pipeline Step 5: run_analysis.py for each REQUIRED_COHORTS (cohort, age_band)
# Set FORCE_STEP5 = True to re-run even when S3 outputs or checkpoints exist
FORCE_STEP5 = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 5: {cohort} / {age_band}")
        cmd = [sys.executable, "run_analysis.py", "--cohort-name", cohort, "--age-band", age_band]
        if FORCE_STEP5:
            cmd.append("--force")
        r = subprocess.run(cmd, cwd=PROJECT_ROOT / "5_pgx_analysis")
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 5 complete.")

→ Step 5: opioid_ed / 13-24
2026-02-10 07:16:51,154 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-10 07:16:51,154 - INFO - Force re-run: ignoring existing S3 outputs and checkpoints.
2026-02-10 07:16:51,154 - INFO - [FUNCTION][5_pgx_analysis][run_pgx_analysis] START time=2026-02-10 07:16:51 mem_mb=1642.8 cpu_pct=0.0
2026-02-10 07:16:51,155 - INFO - Starting PGx analysis for opioid_ed / 13-24
2026-02-10 07:16:51,155 - INFO - Using global drug-to-CPIC mapping from outputs/global/drug_cpic_mapping_global.csv
2026-02-10 07:16:51,155 - INFO - [STEP][5_pgx_analysis][create_pgx_features] START time=2026-02-10 07:16:51 mem_mb=1642.8 cpu_pct=0.0
2026-02-10 07:16:51,155 - INFO - Creating PGx features for opioid_ed / 13-24
2026-02-10 07:16:52,813 - INFO - PGx features created
2026-02-10 07:16:52,813 - INFO - Create stdout:

Created 2 PGx features for 14413 patients
Output format: Ready for merging with other features (uses mi_person_key)
Saved to:

# Pipeline Phase 6: Final model deployment outputs

Train final models per cohort/age_band. Reads Step 4 model data and Step 5 PGx features; writes trained models and `feature_schema.json` to `6_final_model/outputs` (or DATA_ROOT). These outputs are used by "Prepare models" and deployment below.

In [11]:
# Pipeline Step 6: run_final_model.py for each REQUIRED_COHORTS (cohort, age_band)
# Note: script uses --age_band (underscore)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 6: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "run_final_model.py", "--cohort", cohort, "--age_band", age_band],
            cwd=PROJECT_ROOT / "6_final_model",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 6 complete.")

→ Step 6: opioid_ed / 13-24
2026-02-10 07:33:13,055 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-10 07:33:13,157 - INFO - [FUNCTION][final_model][run_final_model] START time=2026-02-10 07:33:13 mem_mb=1300.3 cpu_pct=0.6
2026-02-10 07:33:13,157 - INFO - [STEP][final_model][build_final_features] START time=2026-02-10 07:33:13 mem_mb=1300.3 cpu_pct=0.0
Found model_events.parquet at: /mnt/nvme/4_model_data/cohort_name=opioid_ed/age_band=13-24/model_events.parquet
Loading model data (cases + controls) from /mnt/nvme/4_model_data/cohort_name=opioid_ed/age_band=13-24/model_events.parquet
Class distribution after aggregation:
  Cases (target=1): 2333
  Controls (target=0): 12080

[INFO] Using Step 3b refined feature importance (leakage-filtered): /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outputs/opioid_ed/13_24/opioid_ed_13_24_cohort_feature_importance.csv
[INFO] Loaded 2519 features from Step 3b refined feature importance
[INFO] Re

### Model performance summary and Top 20 feature importance (per cohort)

Before shutting down EC2: print final model performance metrics for all cohorts and Top 20 XGBoost feature importance bar charts per cohort.

In [ ]:
# Model performance metrics (all cohorts) and Top 20 feature importance bar charts per cohort
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Resolve outputs base (project or NVMe)
def _outputs_base():
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        if base and base.exists():
            return base
    return FINAL_MODEL_OUTPUTS

base = _outputs_base()
print("Final model performance by cohort (selected model)")
print("=" * 80)
all_metrics = []
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        path = base / cohort / ab_f / f"{cohort}_{ab_f}_model_metrics_summary.csv"
        if not path.exists():
            print(f"  [skip] {cohort} / {age_band}: no metrics CSV")
            continue
        df = pd.read_csv(path)
        selected = df.loc[df["selected"] == True]
        if selected.empty:
            selected = df.head(1)
        for _, row in selected.iterrows():
            all_metrics.append({
                "cohort": cohort,
                "age_band": age_band,
                "model": row["model"],
                "recall_mean": row["recall_mean"],
                "pr_auc_mean": row["pr_auc_mean"],
                "auc_mean": row.get("auc_mean", None),
                "logloss_mean": row.get("logloss_mean", None),
            })
if all_metrics:
    summary = pd.DataFrame(all_metrics)
    print(summary.to_string(index=False))
else:
    print("  No metrics CSVs found under", base)
print()
print("Top 20 feature importance (XGBoost) per cohort")
print("=" * 80)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        fi_path = base / cohort / ab_f / f"{cohort}_{ab_f}_xgboost_feature_importance.csv"
        if not fi_path.exists():
            print(f"  [skip] {cohort} / {age_band}: no feature importance CSV")
            continue
        fi = pd.read_csv(fi_path).sort_values("importance", ascending=False).head(20)
        if fi.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(range(len(fi)), fi["importance"].values, align="center")
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi["feature"].values, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("Importance (gain)")
        ax.set_title(f"Top 20 features — {cohort} / {age_band}")
        plt.tight_layout()
        plt.show()


### Step 1a: Generate Model Metadata

Extract valid codes (drugs, ICD, CPT) from feature importance for dashboard dropdowns. Uses Step 3b `cohort_feature_importance` when available, else Step 3 aggregated. **Checkpoint:** step is skipped if S3 checkpoint exists. Set `FORCE_STEP1A = True` in the cell below to re-run and rebuild `outputs/metadata/` (e.g. for Prepare Lambda directory). Run this before Step 7 (SHAP) so metadata is ready for deployment.

In [12]:
import logging
logger = logging.getLogger(__name__)
FORCE_STEP1A = True  # Set True to re-run and build outputs/metadata even when checkpoint exists
if not FORCE_STEP1A and check_step_checkpoint_exists("9_dashboard_metadata", "all", "all", logger):
    print("Step 1 (generate metadata) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "generate_metadata.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_metadata", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

Output directory: /home/pgx3874/pgx-analysis/10_risk_dashboard/outputs/metadata

Generating metadata for opioid_ed
Processing opioid_ed / 13-24...
Loading Step 3b refined features: /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outputs/opioid_ed/13_24/opioid_ed_13_24_cohort_feature_importance.csv
  Found 40 drugs, 51 ICDs, 109 CPTs
Processing opioid_ed / 25-44...
Loading Step 3b refined features: /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outputs/opioid_ed/25_44/opioid_ed_25_44_cohort_feature_importance.csv
  Found 48 drugs, 63 ICDs, 89 CPTs
Processing opioid_ed / 45-54...
Loading Step 3b refined features: /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outputs/opioid_ed/45_54/opioid_ed_45_54_cohort_feature_importance.csv
  Found 52 drugs, 66 ICDs, 82 CPTs
Processing opioid_ed / 55-64...
Loading Step 3b refined features: /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outputs/opioid_ed/55_64/opioid_ed_55_64_cohort_feature_importance.csv
  Found 69 drugs, 55 IC

### Step 7: SHAP values

Generate SHAP values for each cohort/age_band using **7_shap_analysis/run_shap_analysis.py**. Outputs go to `7_shap_analysis/outputs/{cohort}/{age_band_fname}/` (global importance CSV, sample parquet). Run this **before** Step 8 (FFA) and Combine.

In [ ]:
# Step 7: Generate SHAP values per cohort/age_band (7_shap_analysis/outputs).
SHAP_SCRIPT = PROJECT_ROOT / "7_shap_analysis" / "run_shap_analysis.py"
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 7 (SHAP): {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, str(SHAP_SCRIPT), "--cohort", cohort, "--age_band", age_band],
            cwd=PROJECT_ROOT,
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 7 (SHAP) complete.")

→ Step 7 (SHAP): opioid_ed / 13-24
[SKIP] Step 7 outputs already exist in S3 for opioid_ed/13-24; downloading to local.
Downloaded /home/pgx3874/pgx-analysis/7_shap_analysis/outputs/opioid_ed/13_24/opioid_ed_13_24_shap_global_importance_xgboost.csv from S3
Downloaded /home/pgx3874/pgx-analysis/7_shap_analysis/outputs/opioid_ed/13_24/opioid_ed_13_24_shap_sample_values_xgboost.parquet from S3
Downloaded /home/pgx3874/pgx-analysis/7_shap_analysis/outputs/opioid_ed/13_24/opioid_ed_13_24_shap_global_importance_catboost.csv from S3
Downloaded /home/pgx3874/pgx-analysis/7_shap_analysis/outputs/opioid_ed/13_24/opioid_ed_13_24_shap_sample_values_catboost.parquet from S3
[WARNING] Required SHAP outputs missing after download attempt. Will regenerate.
Loading final features for opioid_ed, 13-24...
Final feature matrix: 14413 rows, 2520 features.
Loading best models for SHAP...
Loaded best CatBoost model from /home/pgx3874/pgx-analysis/6_final_model/outputs/opioid_ed/13_24/models/catboost_model.cb

### Step 8: FFA rules

Run FFA (Formal Feature Attribution) per cohort/age_band using **run_shap_ffa_workflow.py** with **--skip-shap --skip-combine**: uses existing Step 7 SHAP and XGBoost JSON, writes to `8_ffa_analysis/outputs`. Run this **after** Step 7 and **before** Combine.